# fastText 확장 ESG 사전 세부 threshold 생성

## 분석 목적

이 노트북의 목적은 초기 확장 사전 생성 결과를 보완해, 0.55부터 0.75까지 더 촘촘한 threshold 구간의 ESG 확장 사전을 생성하는 것이다.

## 분석 방법

- seed ESG dictionary를 기준으로 fastText 유사 단어 후보를 다시 산출한다.
- theta 0.55~0.75 구간에서 threshold별 E/S/G 확장 사전을 만든다.
- 이후 확장 사전 검증 노트북에서 threshold별 성능을 비교할 수 있도록 사전 파일을 준비한다.


In [ ]:
from pathlib import Path
import html
import re
import subprocess
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import fasttext
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel", "huggingface_hub"
    ])
    import fasttext
    from huggingface_hub import hf_hub_download

# Colab-friendly Drive mount. Locally, this block is a no-op.
try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

DRIVE_ROOT = Path("/content/drive/MyDrive/UD_26")
LOCAL_ROOT = Path.cwd()

HF_FASTTEXT_REPO_ID = "facebook/fasttext-ko-vectors"
HF_FASTTEXT_FILENAME = "model.bin"
FASTTEXT_TOP_K = 500
THRESHOLDS = [round(theta, 2) for theta in np.arange(0.55, 0.75 + 0.001, 0.01)]
REVIEW_ROWS_PER_THRESHOLD_DIMENSION = 20

SEED_DICTIONARY_CANDIDATES = [
    DRIVE_ROOT / "final" / "seed_dictionary.csv",
    DRIVE_ROOT / "data" / "seed_dictionary.csv",
    LOCAL_ROOT / "final" / "seed_dictionary.csv",
    LOCAL_ROOT / "data" / "seed_dictionary.csv",
    LOCAL_ROOT.parent / "final" / "seed_dictionary.csv",
    LOCAL_ROOT.parent / "data" / "seed_dictionary.csv",
]


def first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {label}. Searched:\n{searched}")


def resolve_output_dir():
    if DRIVE_ROOT.exists():
        return DRIVE_ROOT / "final" / "expanded_dictionaries"
    return (LOCAL_ROOT if (LOCAL_ROOT / "data").exists() else LOCAL_ROOT.parent) / "final" / "expanded_dictionaries"

SEED_DICTIONARY_PATH = first_existing_path(SEED_DICTIONARY_CANDIDATES, "seed_dictionary.csv")
OUTPUT_DIR = resolve_output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_CANDIDATES_PATH = OUTPUT_DIR / "fasttext_raw_candidates_all_seeds.csv"

print("SEED_DICTIONARY_PATH:", SEED_DICTIONARY_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RAW_CANDIDATES_PATH:", RAW_CANDIDATES_PATH)
print("HF_FASTTEXT_REPO_ID:", HF_FASTTEXT_REPO_ID)
print("FASTTEXT_TOP_K:", FASTTEXT_TOP_K)
print("THRESHOLDS:", THRESHOLDS)


In [ ]:
seed_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")

required_seed_columns = {"dimension", "seed_term"}
missing_seed = required_seed_columns - set(seed_df.columns)
if missing_seed:
    raise ValueError(f"seed_dictionary.csv missing columns: {sorted(missing_seed)}")

print("seed_df shape:", seed_df.shape)
display(seed_df.head())


In [ ]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_term(term):
    return normalize_text(term)


def threshold_label(theta):
    return f"{theta:.2f}".replace(".", "_")


def dictionary_name(theta):
    return f"expanded_dictionary_theta_{threshold_label(theta)}"


def pattern_from_term(term):
    return re.escape(normalize_term(term))

seed_df = seed_df.copy()
seed_df["dimension"] = seed_df["dimension"].map(normalize_term)
seed_df["seed_term"] = seed_df["seed_term"].map(normalize_term)
if "pattern" not in seed_df.columns:
    seed_df["pattern"] = ""

seed_df = seed_df[seed_df["dimension"].ne("") & seed_df["seed_term"].ne("")].copy()

print("normalized seed rows:", len(seed_df))
display(seed_df.head())


In [ ]:
def build_seed_query_frame(seed_source_df):
    rows = []
    for _, row in seed_source_df.iterrows():
        values = [row.get("seed_term", "")]
        pattern = row.get("pattern", "")
        if pd.notna(pattern):
            values.extend(str(pattern).split("|"))

        seen = set()
        for value in values:
            query_term = normalize_term(value)
            if not query_term or query_term.lower() == "nan" or query_term in seen:
                continue
            seen.add(query_term)
            rows.append({
                "dimension": row["dimension"],
                "seed_term": normalize_term(row["seed_term"]),
                "query_term": query_term,
            })

    return pd.DataFrame(rows).drop_duplicates(["dimension", "seed_term", "query_term"]).reset_index(drop=True)


seed_query_df = build_seed_query_frame(seed_df)
print("seed query rows:", len(seed_query_df))
display(seed_query_df.groupby("dimension").size().rename("query_terms"))


In [ ]:
model_path = hf_hub_download(repo_id=HF_FASTTEXT_REPO_ID, filename=HF_FASTTEXT_FILENAME)
fasttext_model = fasttext.load_model(model_path)

raw_candidate_rows = []
for _, row in seed_query_df.iterrows():
    query_term = row["query_term"]
    try:
        neighbors = fasttext_model.get_nearest_neighbors(query_term, k=FASTTEXT_TOP_K)
    except Exception as exc:
        print(f"Skipping {query_term!r}: {exc}")
        continue

    for similarity, candidate_term in neighbors:
        candidate_term = normalize_term(candidate_term)
        if not candidate_term or candidate_term == query_term:
            continue
        raw_candidate_rows.append({
            "dimension": row["dimension"],
            "seed_term": row["seed_term"],
            "query_term": query_term,
            "candidate_term": candidate_term,
            "similarity": float(similarity),
        })

raw_candidates_df = pd.DataFrame(raw_candidate_rows)
if raw_candidates_df.empty:
    raise ValueError("No fastText candidates were generated. Check seed terms and model loading.")

raw_candidates_df = raw_candidates_df.drop_duplicates(
    ["dimension", "seed_term", "query_term", "candidate_term"]
).sort_values(["dimension", "seed_term", "query_term", "similarity"], ascending=[True, True, True, False])
raw_candidates_df.to_csv(RAW_CANDIDATES_PATH, index=False, encoding="utf-8-sig")

print("raw candidate rows:", len(raw_candidates_df))
print("saved:", RAW_CANDIDATES_PATH)
display(raw_candidates_df.head(20))


In [ ]:
def build_seed_dictionary_frame(seed_source_df):
    rows = []
    for _, row in seed_source_df.iterrows():
        values = [row.get("seed_term", "")]
        pattern = row.get("pattern", "")
        if pd.notna(pattern):
            values.extend(str(pattern).split("|"))

        seen = set()
        for value in values:
            term = normalize_term(value)
            if not term or term.lower() == "nan" or term in seen:
                continue
            seen.add(term)
            rows.append({
                "dictionary_name": "seed_dictionary",
                "threshold": np.nan,
                "dimension": row["dimension"],
                "seed_term": normalize_term(row["seed_term"]),
                "candidate_term": term,
                "pattern": pattern_from_term(term),
                "similarity": 1.0,
                "source": "seed",
                "seed_terms_matched": normalize_term(row["seed_term"]),
                "query_terms_matched": term,
                "keep_review": True,
                "exclude_reason": "",
            })

    seed_dictionary = pd.DataFrame(rows)
    return seed_dictionary.drop_duplicates(["dimension", "candidate_term"]).reset_index(drop=True)


def build_expanded_dictionary_for_threshold(theta):
    seed_rows = build_seed_dictionary_frame(seed_df)
    seed_rows["dictionary_name"] = dictionary_name(theta)
    seed_rows["threshold"] = theta

    filtered = raw_candidates_df.loc[raw_candidates_df["similarity"].ge(theta)].copy()

    grouped_rows = []
    group_columns = ["dimension", "candidate_term"]
    for (dimension, candidate_term), group in filtered.groupby(group_columns, sort=False):
        best = group.sort_values("similarity", ascending=False).iloc[0]
        seed_terms = "; ".join(sorted(group["seed_term"].dropna().unique()))
        query_terms = "; ".join(sorted(group["query_term"].dropna().unique()))
        grouped_rows.append({
            "dictionary_name": dictionary_name(theta),
            "threshold": theta,
            "dimension": dimension,
            "seed_term": best["seed_term"],
            "candidate_term": candidate_term,
            "pattern": pattern_from_term(candidate_term),
            "similarity": float(best["similarity"]),
            "source": "fasttext_candidate",
            "seed_terms_matched": seed_terms,
            "query_terms_matched": query_terms,
            "keep_review": "REVIEW",
            "exclude_reason": "",
        })

    candidate_df = pd.DataFrame(grouped_rows)
    out_df = pd.concat([seed_rows, candidate_df], ignore_index=True)
    out_df = out_df.drop_duplicates(["dimension", "candidate_term"], keep="first")
    return out_df.sort_values(
        ["dimension", "source", "similarity", "candidate_term"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)

seed_dictionary_df = build_seed_dictionary_frame(seed_df)
print("seed dictionary rows:", len(seed_dictionary_df))
display(seed_dictionary_df.groupby("dimension").size().rename("seed_terms"))


In [ ]:
seed_dictionary_path = OUTPUT_DIR / "seed_dictionary_normalized_for_comparison.csv"
seed_dictionary_df.to_csv(seed_dictionary_path, index=False, encoding="utf-8-sig")

expanded_dictionary_map = {}
saved_files = [seed_dictionary_path, RAW_CANDIDATES_PATH]
summary_rows = []

for theta in THRESHOLDS:
    name = dictionary_name(theta)
    expanded_df = build_expanded_dictionary_for_threshold(theta)
    expanded_dictionary_map[name] = expanded_df

    out_path = OUTPUT_DIR / f"{name}.csv"
    expanded_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    saved_files.append(out_path)

    counts = (
        expanded_df.groupby(["dimension", "source"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for source_col in ["seed", "fasttext_candidate"]:
        if source_col not in counts.columns:
            counts[source_col] = 0
    counts["threshold"] = theta
    counts["dictionary_name"] = name
    counts["total_terms"] = counts["seed"] + counts["fasttext_candidate"]
    summary_rows.append(counts[["dictionary_name", "threshold", "dimension", "seed", "fasttext_candidate", "total_terms"]])

summary_df = pd.concat(summary_rows, ignore_index=True)
summary_path = OUTPUT_DIR / "expanded_dictionary_threshold_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
saved_files.append(summary_path)

print("saved files:")
for path in saved_files:
    print("-", path)

display(summary_df)


In [ ]:
candidate_counts = summary_df.pivot_table(
    index="threshold",
    columns="dimension",
    values="fasttext_candidate",
    aggfunc="sum",
    fill_value=0,
).astype(int)

term_counts = summary_df.pivot_table(
    index="threshold",
    columns="dimension",
    values="total_terms",
    aggfunc="sum",
    fill_value=0,
).astype(int)

print("fastText candidate counts by threshold/dimension")
display(candidate_counts)
print("total dictionary terms by threshold/dimension")
display(term_counts)


In [ ]:
review_frames = []
for theta, name in [(theta, dictionary_name(theta)) for theta in THRESHOLDS]:
    candidates = expanded_dictionary_map[name]
    candidates = candidates[candidates["source"].eq("fasttext_candidate")].copy()
    candidates["threshold"] = theta
    review_frames.append(candidates)

manual_review_df = pd.concat(review_frames, ignore_index=True) if review_frames else pd.DataFrame()
manual_review_columns = [
    "threshold", "dimension", "candidate_term", "similarity", "seed_terms_matched",
    "query_terms_matched", "keep_review", "exclude_reason",
]
manual_review_df = manual_review_df[manual_review_columns].sort_values(
    ["threshold", "dimension", "similarity", "candidate_term"],
    ascending=[True, True, True, True],
).reset_index(drop=True)

manual_review_sample = (
    manual_review_df.groupby(["threshold", "dimension"], group_keys=False)
    .head(REVIEW_ROWS_PER_THRESHOLD_DIMENSION)
    .reset_index(drop=True)
)

print("manual_review_df rows:", len(manual_review_df))
print("displaying up to", REVIEW_ROWS_PER_THRESHOLD_DIMENSION, "rows per threshold/dimension")
display(manual_review_sample)
